In [ ]:
input_file_gap = '/kaggle/output/submission.csv'
output_file_gap = '/kaggle/output/clean_submission.csv'
diacritic_pattern = re.compile(r'([\u09BE-\u09CC])\1+')

def remove_malformed_words(text):
    """Removes words entirely if they contain consecutive identical diacritics."""
    if not isinstance(text, str):
        return text
    
    words = text.split()
    valid_words = [w for w in words if not diacritic_pattern.search(w)]
    return ' '.join(valid_words)

def remove_repetitions_with_gap(text, max_gap_words=3):
    if not isinstance(text, str): return text
    
    text = remove_malformed_words(text)
    
    words = text.split()
    if not words: return text
    
    has_changed = True
    limit_n = 4 
    
    while has_changed:
        has_changed = False
        
        for n in range(limit_n, 0, -1):
            i = 0
            new_words = []
            
            
            while i < len(words):
                chunk = words[i : i+n]
                
                
                match_found = False
                match_start_index = -1
                
                search_start = i + n
                search_end = min(len(words), search_start + max_gap_words + 1)
                
                for j in range(search_start, search_end):
                    if j + n <= len(words):
                        if words[j : j+n] == chunk:
                            match_found = True
                            match_start_index = j
                            break 
                
                if match_found:
                    # Logic: 
                    # Structure: [Original Chunk] [Gap Words] [Repetition]
                    # Indices:   i                i+n         match_start
                    # Action: Keep [Original Chunk] + [Gap Words]. Skip [Repetition].
                    
                    # 1. Add everything from current 'i' up to 'match_start_index'
                    # This includes the chunk at 'i' and the gap words.
                    new_words.extend(words[i : match_start_index])
                    
                    # 2. Skip the recurring chunk at 'match_start_index'
                    i = match_start_index + n
                    
                    has_changed = True
                else:
                    new_words.append(words[i])
                    i += 1
            
            if has_changed:
                words = new_words
                break
                
    return ' '.join(words)

try:
    df_gap = pd.read_csv(input_file_gap)
    print(f"Loaded {input_file_gap}")
except FileNotFoundError:
    print(f"File {input_file_gap} not found.")
    df_gap = pd.DataFrame()

if 'transcript' in df_gap.columns:
    df_gap['transcript'] = df_gap['transcript'].apply(lambda x: remove_repetitions_with_gap(x, max_gap_words=3))
    df_gap.to_csv(output_file_gap, index=False)
    print(f"Saved enhanced clean file to {output_file_gap}")
    
    test_1 = "সেন্টার সেন্টাা"
    test_2 = "বিভিন্ন প্রেসিডেন্টের প্রেসিডেন্ট বিভিন্ন প্রেসিডেন্টের"
    test_3 = "আমি আমি যাব আমি" 
    
    print("\n--- Tests ---")
    print(f"Input: {test_1} -> Output: {remove_repetitions_with_gap(test_1)}")
    print(f"Input: {test_2} -> Output: {remove_repetitions_with_gap(test_2)}")
    print(f"Input: {test_3} -> Output: {remove_repetitions_with_gap(test_3)}")
